# fastsafetensors smoke test

fastsafetensors reads model weight files with several threads straight into the target device's memory (vLLM's `--load-format fastsafetensors`). Its C++ extension links none of CUDA, cuFile or libnuma: it opens `libcudart.so`, `libcufile.so.0` and `libnuma.so.1` at run time, and loads onto the CPU without them. Examples 1-3 and 5 run on the CPU; Example 4 loads onto the GPU. No network is needed: the weight files are written by the cells themselves.

## Example 1 — the import, the version and what the C++ extension found at run time

In [ ]:
import importlib.metadata
import fastsafetensors
import fastsafetensors.cpp as extension

print("fastsafetensors", fastsafetensors.__version__)
assert fastsafetensors.__version__ == importlib.metadata.version("fastsafetensors")
assert extension.__file__.endswith(".so"), extension.__file__
for name in ("SafeTensorsFileLoader", "ParallelLoader", "SingleGroup", "fastsafe_open"):
    assert hasattr(fastsafetensors, name), name  # the names vLLM imports
assert extension.get_alignment_size() == 4096, extension.get_alignment_size()
assert not extension.is_cuda_found()  # nothing is opened at import ...
extension.load_library_functions()  # ... this is what the first loader does: try libcudart, libcufile, libnuma
found_cuda, found_cufile = extension.is_cuda_found(), extension.is_cufile_found()
# On a machine with no NVIDIA driver the CUDA runtime cannot start, both read False, and the loader reads onto the
# CPU (Examples 2, 3 and 5). Example 4, which needs a GPU, asserts that the CUDA runtime is found there.
assert isinstance(found_cuda, bool) and isinstance(found_cufile, bool), (found_cuda, found_cufile)
print(f"OK -- extension loaded; trying the run-time libraries raised nothing; CUDA runtime usable: {found_cuda}, cuFile: {found_cufile}")

## Example 2 — load two weight files on the CPU; every tensor equals what was saved

In [ ]:
import os, tempfile, torch
from safetensors.torch import save_file
from fastsafetensors import fastsafe_open

torch.manual_seed(0)
folder = tempfile.mkdtemp(prefix="fastsafetensors_nb_")
saved = {"layer.weight": torch.randn(64, 32), "token_ids": torch.arange(10, dtype=torch.int64),
         "half": torch.randn(8, 8).to(torch.bfloat16)}
first, second = os.path.join(folder, "model-00001.safetensors"), os.path.join(folder, "model-00002.safetensors")
save_file({k: saved[k] for k in ("layer.weight", "token_ids")}, first)
save_file({"half": saved["half"]}, second)
with fastsafe_open([first, second], framework="pt", device="cpu", nogds=True) as files:
    assert sorted(files.keys()) == sorted(saved), sorted(files.keys())
    for name, expected in saved.items():
        loaded = files.get_tensor(name)
        assert loaded.dtype == expected.dtype and loaded.shape == expected.shape, (name, loaded.dtype, loaded.shape)
        assert torch.equal(loaded, expected), f"{name} differs from what was saved"
print("OK -- float32, int64 and bfloat16 tensors from two files load equal to what was saved")

## Example 3 — the loader vLLM drives: files grouped by rank, copied once, read by name

In [ ]:
import os, tempfile, torch
from safetensors.torch import save_file
from fastsafetensors import ParallelLoader, SafeTensorsFileLoader, SingleGroup

torch.manual_seed(1)
folder = tempfile.mkdtemp(prefix="fastsafetensors_nb_")
saved = {"a": torch.randn(128, 16), "b": torch.randint(0, 100, (33,), dtype=torch.int32)}
path = os.path.join(folder, "weights.safetensors")
save_file(saved, path)
loader = SafeTensorsFileLoader(SingleGroup(), device="cpu", nogds=True)
loader.add_filenames({0: [path]})  # rank 0 gets this file
buffers = loader.copy_files_to_device()
assert torch.equal(buffers.get_tensor("a"), saved["a"]) and torch.equal(buffers.get_tensor("b"), saved["b"])
assert buffers.get_filename("a") == path, buffers.get_filename("a")
buffers.close()
loader.close()
parallel = ParallelLoader(SingleGroup(), [path], device="cpu", nogds=True, use_tqdm_on_load=False)
streamed = {name: tensor.clone() for name, tensor in parallel.iterate_weights()}  # vLLM's own loop
assert sorted(streamed) == ["a", "b"] and all(torch.equal(streamed[k], saved[k]) for k in saved)
print("OK -- the low-level loader and ParallelLoader.iterate_weights() both return the saved tensors")

## Example 4 — load straight onto the GPU

In [ ]:
import os, tempfile, torch
from safetensors.torch import save_file
from fastsafetensors import fastsafe_open
import fastsafetensors.cpp as extension

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
assert extension.is_cuda_found(), "the CUDA runtime was not found"
print("cuFile found:", extension.is_cufile_found())  # not needed here: nogds=True reads with plain file I/O
torch.manual_seed(2)
saved = {"weight": torch.randn(256, 64), "bias": torch.randn(64).to(torch.float16)}
path = os.path.join(tempfile.mkdtemp(prefix="fastsafetensors_nb_"), "gpu.safetensors")
save_file(saved, path)
with fastsafe_open([path], framework="pt", device="cuda:0", nogds=True) as files:
    for name, expected in saved.items():
        loaded = files.get_tensor(name)
        assert loaded.device.type == "cuda" and loaded.dtype == expected.dtype, (name, loaded.device, loaded.dtype)
        assert torch.equal(loaded.cpu(), expected), f"{name} differs on the device"
print("OK -- two tensors loaded onto cuda:0 equal what was saved (plain file reads, no GPUDirect Storage)")

## Example 5 — a missing, a truncated and a non-safetensors file are each refused with an error

In [ ]:
import os, tempfile, torch
from safetensors.torch import save_file
from fastsafetensors import fastsafe_open

folder = tempfile.mkdtemp(prefix="fastsafetensors_nb_")
good = os.path.join(folder, "good.safetensors")
save_file({"w": torch.randn(64, 32)}, good)
with open(good, "rb") as handle:
    content = handle.read()
with open(os.path.join(folder, "truncated.safetensors"), "wb") as handle:
    handle.write(content[: len(content) // 2])
with open(os.path.join(folder, "junk.safetensors"), "wb") as handle:
    handle.write(b"not a safetensors file at all")
refusals = {}
for name in ("missing.safetensors", "truncated.safetensors", "junk.safetensors"):
    try:
        with fastsafe_open([os.path.join(folder, name)], device="cpu", nogds=True) as files:
            files.get_tensor("w")
        raise AssertionError(f"{name} was loaded without an error")
    except AssertionError:
        raise
    except Exception as error:
        refusals[name] = f"{type(error).__name__}: {str(error).split(',')[0][-60:]}"
assert refusals["missing.safetensors"].startswith("FileNotFoundError"), refusals
assert "MetadataIncompleteBuffer" in refusals["truncated.safetensors"] and "HeaderTooLarge" in refusals["junk.safetensors"], refusals
print("OK -- refused:", refusals)